# Detector 完整实验入口
方法、环境与限制详见 README.md。所有输出限定在 detector。默认只打印 GPU 实验命令，避免 Run All 意外启动长任务。

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

proact_root = next(
    p
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "main_baselines.py").is_file() and (p / "detector").is_dir()
)
os.chdir(proact_root)
env = dict(os.environ, PYTHONDONTWRITEBYTECODE="1")


def run(*args):
    subprocess.run([sys.executable, "-B", *args], cwd=proact_root, env=env, check=True)


print(proact_root)
RUN_BOOTSTRAP = False
RUN_EXPERIMENT = False

## 1. CPU 单元测试
不下载数据，不生成论文指标。

In [ ]:
run("-m", "unittest", "discover", "-s", "detector/tests", "-v")

## 2. 可选：生成 victim、反演、攻击与效果验证
已有材料时跳过。需要匹配的 CUDA/torch/torchvision 环境，默认 dry-run。

In [ ]:
run("-m", "detector.bootstrap", *([] if RUN_BOOTSTRAP else ["--dry-run"]))

## 3. 编辑 config.example.json 后运行完整流程
路径相对于配置文件目录，data_cwd 下需要 data/cifar-100-python。开启 RUN_EXPERIMENT 才执行。

In [ ]:
run(
    "-m",
    "detector.pipeline",
    "--config",
    "detector/config.example.json",
    *([] if RUN_EXPERIMENT else ["--dry-run"]),
)

## 4. 查看结果
默认报告：detector/work/full_v2_seed0/report.md。一起看 clean 误拒、poison 检出、random-control 告警；无标签 p-value 不是投毒概率。独立 incoming dataset 的预测命令见 README 第 6 节。